## 1. Install Dependencies

In [ ]:
# Install dependencies (run once)
# Using CUDA-enabled PyTorch for maximum GPU performance
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q ctranslate2 transformers sentencepiece accelerate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


## 2. Configuration

In [ ]:
import os
import re
import time
from typing import List
import torch

# ============================================================================
# CONFIGURATION - Edit these paths
# ============================================================================

# Input file: Source-Target pairs (TAB separated)
INPUT_FILE = "../../0_data/processed/en_tgt_pairs.txt"

# Output file: Pivot-Target pairs (TAB separated)
OUTPUT_FILE = "../../0_data/processed/fr_tgt_pairs_translated.txt"

# ============================================================================
# GPU PERFORMANCE SETTINGS
# ============================================================================

# Detect GPU and get available memory
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory_gb = torch.cuda.get_device_properties(
        0).total_memory / (1024**3)
    print(f"🚀 GPU detected: {gpu_name}")
    print(f"   Total memory: {gpu_memory_gb:.1f} GB")

    # Auto-adjust batch size based on GPU memory
    if gpu_memory_gb >= 24:      # RTX 3090, RTX 4090, A10, etc.
        BATCH_SIZE = 128
    elif gpu_memory_gb >= 16:    # T4, RTX 4080, etc.
        BATCH_SIZE = 96
    elif gpu_memory_gb >= 12:    # RTX 3060 12GB
        BATCH_SIZE = 64
    elif gpu_memory_gb >= 10:    # RTX 3080 10GB
        BATCH_SIZE = 48
    elif gpu_memory_gb >= 8:     # RTX 3070, RTX 4060
        BATCH_SIZE = 32
    else:
        BATCH_SIZE = 16

    DEVICE = "cuda"
else:
    print("⚠️ No GPU detected, using CPU (will be slower)")
    BATCH_SIZE = 8
    DEVICE = "cpu"

# OVERRIDE: Set manual batch size here if you want (uncomment)
# BATCH_SIZE = 64

# Number of parallel translation workers (use more for bigger GPUs)
# More workers = more GPU utilization, but more VRAM usage
INTER_THREADS = 4   # Parallel batch processing
INTRA_THREADS = 4   # Parallel ops within each batch

# Use FP16 compute (faster on modern GPUs with Tensor Cores)
# Set to "default" if you have issues, or "float16" for max speed on RTX/A100
# Options: "int8" (fast+small), "float16" (faster on RTX), "int8_float16" (best balance)
COMPUTE_TYPE = "int8"

# Model settings
MODEL_NAME = "facebook/nllb-200-distilled-600M"
CT2_MODEL_PATH = "../models/ct2fast-nllb-200-distilled-600M"

# Separator pattern
TAB_PATTERN = re.compile(r'\t+')

# Language codes (NLLB format)
LANG_CODES = {
    "en": "eng_Latn",
    "fr": "fra_Latn",
    "kab": "tgt_Latn",
}

print(f"\n✓ Configuration loaded!")
print(f"  Input: {INPUT_FILE}")
print(f"  Output: {OUTPUT_FILE}")
print(f"  Device: {DEVICE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Compute type: {COMPUTE_TYPE}")
print(f"  Threads: {INTER_THREADS} inter, {INTRA_THREADS} intra")

Configuration loaded!
  Input: en_tgt_pairs.txt
  Output: fr_tgt_pairs_translated.txt
  Device: cuda
  Batch size: 32


## 3. Download Model

In [3]:
import shutil
import subprocess
import sys


def convert_model_to_ct2(model_name: str, output_path: str):
    """Convert HuggingFace model to CTranslate2 format."""
    print(f"Converting model {model_name} to CTranslate2 format...")
    print("This only needs to be done once and may take a few minutes...")

    result = subprocess.run([
        sys.executable, "-m", "ctranslate2.converters.transformers",
        "--model", model_name,
        "--output_dir", output_path,
        "--quantization", "int8",
        "--force"
    ], capture_output=True, text=True)

    if result.returncode != 0:
        print(f"Error: {result.stderr}")
        raise RuntimeError("Model conversion failed")

    print(f"✓ Model saved to {output_path}")


# Check if model exists and has required files
model_bin = os.path.join(CT2_MODEL_PATH, "model.bin")
if not os.path.exists(model_bin):
    if os.path.exists(CT2_MODEL_PATH):
        shutil.rmtree(CT2_MODEL_PATH)  # Remove incomplete model
    convert_model_to_ct2(MODEL_NAME, CT2_MODEL_PATH)
else:
    print(f"✓ Model already exists at {CT2_MODEL_PATH}")

Converting model facebook/nllb-200-distilled-600M to CTranslate2 format...
This only needs to be done once and may take a few minutes...
Error: <frozen runpy>:128: RuntimeWarning: 'ctranslate2.converters.transformers' found in sys.modules after import of package 'ctranslate2.converters', but prior to execution of 'ctranslate2.converters.transformers'; this may result in unpredictable behaviour
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "d:\Documents\py\projects\chat_aqvayli\.venv\Lib\site-packages\ctranslate2\converters\transformers.py", line 3196, in <module>
    main()
  File "d:\Documents\py\projects\chat_aqvayli\.venv\Lib\site-packages\ctranslate2\converters\transformers.py", line 3192, in main
    converter.convert_from_args(args)
  File "d:\Documents\py\projects\chat_aqvayli\.venv\Lib\site-packages\ctranslate2\converters\converter.py", line 50, in convert_from_args
    return s

RuntimeError: Model conversion failed

## 4. Load Model & Tokenizer

In [ ]:
import ctranslate2
import transformers

print(f"Loading model from {CT2_MODEL_PATH}...")
print(f"Settings: device={DEVICE}, compute={COMPUTE_TYPE}, batch={BATCH_SIZE}")

# Load CTranslate2 model with max performance settings
translator = ctranslate2.Translator(
    CT2_MODEL_PATH,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,        # int8/float16/int8_float16
    inter_threads=INTER_THREADS,       # Parallel batch processing
    intra_threads=INTRA_THREADS,       # Parallel ops per batch
)

# Load tokenizer
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"✓ Model loaded on {DEVICE.upper()}")
print(f"✓ Ready for high-speed translation!")

## 5. Translation Functions

In [ ]:
def translate_batch_sentences(
    sentences: List[str],
    src_lang: str,
    tgt_lang: str,
) -> List[str]:
    """Translate a batch of sentences with maximum GPU utilization."""

    # Set source language
    tokenizer.src_lang = src_lang

    # Tokenize all sentences
    inputs = [tokenizer.convert_ids_to_tokens(
        tokenizer.encode(sent)) for sent in sentences]

    # Get target language token (compatible with all transformers versions)
    tgt_token = tokenizer.convert_ids_to_tokens(
        tokenizer.convert_tokens_to_ids(tgt_lang))
    tgt_prefix = [[tgt_token]]

    # Translate with max performance settings
    results = translator.translate_batch(
        inputs,
        target_prefix=tgt_prefix * len(inputs),
        beam_size=1,              # Greedy decoding (fastest)
        max_batch_size=0,         # No limit, use full batch
        batch_type="tokens",      # Batch by tokens for better GPU utilization
        max_input_length=512,     # Limit input length
        max_decoding_length=512,  # Limit output length
        replace_unknowns=True,    # Handle unknown tokens
    )

    # Decode results
    translations = []
    for result in results:
        tokens = result.hypotheses[0][1:]  # Skip language token
        translation = tokenizer.decode(tokenizer.convert_tokens_to_ids(tokens))
        translations.append(translation)

    return translations


print("✓ Translation functions defined (optimized for max GPU speed)")

## 6. Test Translation

In [ ]:
# Quick test
test_sentences = [
    "Hello, how are you?",
    "The weather is beautiful today.",
    "I love learning new languages.",
]

print("Testing High-Resource Content -> Pivot Language translation:\n")

translations = translate_batch_sentences(
    test_sentences,
    src_lang=LANG_CODES["en"],
    tgt_lang=LANG_CODES["fr"],
)

for en, fr in zip(test_sentences, translations):
    print(f"EN: {en}")
    print(f"FR: {fr}")
    print()

## 7. Load Input File

In [ ]:
# Read input file and parse pairs
print(f"Reading {INPUT_FILE}...")

source_sentences = []
target_sentences = []
skipped = 0

with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = TAB_PATTERN.split(line)
        if len(parts) >= 2:
            source_sentences.append(parts[0].strip())
            target_sentences.append(parts[1].strip())
        else:
            skipped += 1

total = len(source_sentences)
print(f"✓ Found {total:,} pairs to translate")
if skipped > 0:
    print(f"  Skipped {skipped:,} malformed lines")

# Show sample
print("\nSample pairs:")
for i in range(min(3, total)):
    print(f"  EN: {source_sentences[i][:60]}...")
    print(f"  KAB: {target_sentences[i][:60]}...")
    print()

## 8. Run Translation

In [ ]:
# Translate all High-Resource Content sentences to Pivot Language
src_lang = LANG_CODES["en"]
tgt_lang = LANG_CODES["fr"]

print(f"Translating {total:,} sentences: High-Resource Content -> Pivot Language")
print(f"Batch size: {BATCH_SIZE}")
print("-" * 50)

pivot_sentences = []
start_time = time.time()

for i in range(0, total, BATCH_SIZE):
    batch = source_sentences[i:i + BATCH_SIZE]
    batch_translations = translate_batch_sentences(batch, src_lang, tgt_lang)
    pivot_sentences.extend(batch_translations)

    # Progress
    processed = min(i + BATCH_SIZE, total)
    elapsed = time.time() - start_time
    speed = processed / elapsed if elapsed > 0 else 0
    eta = (total - processed) / speed if speed > 0 else 0

    if processed % (BATCH_SIZE * 10) == 0 or processed == total:
        print(f"Progress: {processed:,}/{total:,} ({processed/total*100:.1f}%) | "
              f"Speed: {speed:.1f} sent/sec | ETA: {eta/60:.1f} min")

elapsed = time.time() - start_time
print("\n" + "=" * 50)
print("TRANSLATION COMPLETE!")
print("=" * 50)
print(f"Total pairs: {total:,}")
print(f"Total time: {elapsed/60:.1f} minutes")
print(f"Average speed: {total/elapsed:.1f} sentences/second")

## 9. Save Results

In [ ]:
# Save Pivot-Target pairs
print(f"Saving Pivot-Target pairs to {OUTPUT_FILE}...")

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for french, kabyle in zip(pivot_sentences, target_sentences):
        f.write(f"{french}\t{kabyle}\n")

print(f"✓ Saved {len(pivot_sentences):,} pairs to {OUTPUT_FILE}")

## 10. Verify Results

In [ ]:
# Show sample of translated pairs
print("Sample of translated pairs:\n")
print("=" * 80)

for i in range(min(5, len(pivot_sentences))):
    print(f"\n[{i+1}]")
    print(f"  EN:  {source_sentences[i]}")
    print(f"  FR:  {pivot_sentences[i]}")
    print(f"  KAB: {target_sentences[i]}")

print("\n" + "=" * 80)
print(f"\nOutput file: {OUTPUT_FILE}")